# 18 — GRPO and RLVR with Verifiable Network Rewards

**Network LLM Engineering — Part IV — Post-Training**

### Learning goals
- Understand rollouts, rewards and group-relative advantages
- Build a deterministic subnet reward
- See why RLVR is attractive for verifiable networking tasks

In [ ]:
%pip install -q transformers==5.14.1 datasets==5.0.1 accelerate==1.14.0 peft==0.20.0 trl==1.10.0 sentence-transformers==5.7.0 pandas matplotlib scikit-learn requests jsonschema

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineering_course")]:
        if (p / "data" / "glossary.csv").exists():
            return p
    raise FileNotFoundError("Run from the extracted network_llm_engineering_course folder.")

ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## GRPO

For each prompt, the policy generates a **group** of candidate completions.
A reward function/model scores them. GRPO computes relative advantages within/around the group and updates the policy.

## RLVR

**Reinforcement Learning with Verifiable Rewards** means the reward can be checked objectively.
Networking has valuable verifiable subproblems:
- subnet calculations,
- JSON/schema compliance,
- config syntax parsing,
- route-policy invariants,
- topology reachability in a digital twin,
- unit tests for automation code.

In [ ]:
import ipaddress, json

def subnet_reward(completions, cidr, **kwargs):
    """Return one deterministic reward per completion.

    TRL forwards additional dataset columns (here `cidr`) as batched lists.
    The function works with both standard string completions and conversational completions.
    """
    rewards = []
    for completion, cidr_value in zip(completions, cidr):
        n = ipaddress.ip_network(cidr_value, strict=False)
        expected = {
            "network": str(n.network_address),
            "broadcast": str(n.broadcast_address),
            "prefixlen": n.prefixlen,
            "usable_hosts": max(0, n.num_addresses - 2),
        }
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        try:
            obj = json.loads(text)
            rewards.append(1.0 if all(obj.get(k) == v for k, v in expected.items()) else 0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

good = ['{"network":"192.0.2.64","broadcast":"192.0.2.127","prefixlen":26,"usable_hosts":62}']
print(subnet_reward(good, cidr=["192.0.2.64/26"]))

In [ ]:
from datasets import load_dataset
from trl import GRPOConfig, GRPOTrainer

tasks = load_dataset("json", data_files=str(DATA/"rlvr_subnet_tasks.jsonl"), split="train")
cfg = GRPOConfig(
    output_dir=str(ROOT/"artifacts"/"network-grpo"),
    max_steps=5,
    per_device_train_batch_size=1,
    num_generations=4,
    max_completion_length=100,
    report_to="none",
)
print(tasks)
print(cfg)

# GPU lab:
# trainer = GRPOTrainer(
#     model="Qwen/Qwen3-0.6B",
#     args=cfg,
#     reward_funcs=subnet_reward,
#     train_dataset=tasks,
# )
# trainer.train()

## Critical limitation

Not every network problem has a clean reward.
"Best root cause" under incomplete evidence is not as objectively verifiable as subnet math.
For diagnosis, digital-twin tests, formal constraints, or expert review can create stronger rewards than superficial keyword scoring.

### Exercise

Design a verifiable reward for one of:
- route reachability,
- ACL intent,
- config idempotency,
- JSON change-plan schema,
- digital-twin convergence.

Explain how the reward could still be gamed.